# 01 — Web Scraping: Papal Encyclicals Corpus

**DS 5001 — Exploratory Text Analytics Final Project**  
**Source:** https://www.papalencyclicals.net/document-directory

This notebook walks through scraping the papal encyclicals corpus from
papalencyclicals.net. The site organizes documents by pope, with each
encyclical linked to a page containing the full text (usually HTML, sometimes epub).

## Steps
1. Scrape the document directory to build an index of all encyclicals
2. Download the full text of each encyclical
3. Save raw text files and metadata

In [4]:
import sys
sys.path.insert(0, '..')

import os
os.environ["SCRAPER_ALLOW_INSECURE_SSL"] = "true"

import importlib
import src.scraper as scraper
scraper = importlib.reload(scraper)  # always pick up latest scraper.py edits

# Rebind commonly used symbols from the reloaded module
scrape_index = scraper.scrape_index
scrape_documents = scraper.scrape_documents
save_index = scraper.save_index
save_library_csv = scraper.save_library_csv
fetch_page = scraper.fetch_page
parse_directory = scraper.parse_directory
load_dead_link_replacements = scraper.load_dead_link_replacements
apply_dead_link_replacements = scraper.apply_dead_link_replacements
INDEX_FILE = scraper.INDEX_FILE
RAW_DIR = scraper.RAW_DIR
DATA_DIR = scraper.DATA_DIR
DEAD_LINK_REPLACEMENTS_FILE = scraper.DEAD_LINK_REPLACEMENTS_FILE

import json
import pandas as pd

In [5]:
# Check optional dependencies
import importlib.util

def _check_dep(package, install_cmd):
    if importlib.util.find_spec(package) is None:
        print(f"'{package}' is NOT installed — some features will be disabled.")
        print(f"   To enable: {install_cmd}")
    else:
        print(f"'{package}' is installed.")

_check_dep("lxml",     "pip install lxml")
_check_dep("ebooklib", "pip install ebooklib")


'lxml' is installed.
'ebooklib' is installed.


## Step 1: Scrape the Document Directory

The directory page lists encyclicals organized by pope. We parse it to
extract document titles, URLs, and associated pope names.

In [6]:
# Scrape the directory index (or load if already scraped)
# Set to True after scraper logic changes to rebuild clean metadata
FORCE_REBUILD_INDEX = False

if INDEX_FILE.exists() and not FORCE_REBUILD_INDEX:
    with open(INDEX_FILE) as f:
        documents = json.load(f)
    print(f"Loaded existing index: {len(documents)} documents")
else:
    documents = scrape_index()
    save_index(documents)
    print(f"Scraped index: {len(documents)} documents")

Loaded existing index: 570 documents


In [7]:
# Preview the index
import numpy as np

df_index = pd.DataFrame(documents)

# Backfill normalized fields for legacy index rows loaded from disk.
if 'category' not in df_index.columns:
    df_index['category'] = np.where(
        df_index.get('pope', '').fillna('').str.lower().eq('church councils'),
        'council',
        'pope'
    )
if 'author' not in df_index.columns:
    df_index['author'] = np.where(
        df_index['category'].eq('council'),
        df_index.get('title', ''),
        df_index.get('pope', '')
    )
if 'author_dates' not in df_index.columns:
    df_index['author_dates'] = np.where(
        df_index['category'].eq('council'),
        '',
        df_index.get('pope_dates', '')
    )

print('Documents by category:')
print(df_index['category'].value_counts())

print('\nTop authors:')
print(df_index['author'].value_counts().head(20))

# Years can be empty strings or malformed; coerce to numeric before range stats.
years = pd.to_numeric(df_index['year'], errors='coerce')
if years.notna().any():
    print(f"\nYears covered (valid only): {int(years.min())} - {int(years.max())}")
else:
    print('\nYears covered (valid only): n/a')

print(f"Missing/invalid year values: {years.isna().sum()} of {len(df_index)}")

Documents by category:
category
pope       549
council     21
Name: count, dtype: int64

Top authors:
author
Pope Leo XIII            88
Pope Pius XII            60
Pope St. John Paul II    60
Pope Benedict XIV        44
Pope Bl. Pius IX         42
Pope Paul VI             36
Pope Pius XI             32
Pope St. Pius X          26
Pope Pius VI             26
Pope Benedict XVI        21
Pope St. John XXIII      18
Pope Clement XIII        13
Pope Benedict XV         12
Pope Gregory XVI         10
Pope Francis             10
Pope Leo XII              5
Pope Alexander IV         4
Pope Clement XIV          4
Pope John XXII            4
Pope St. Pius V           4
Name: count, dtype: int64

Years covered (valid only): 1215 - 2016
Missing/invalid year values: 42 of 570


In [8]:
# Audit source domains to troubleshoot external links (e.g., digilander)
from urllib.parse import urlparse

df_domains = df_index.copy()
df_domains['domain'] = df_domains['url'].fillna('').apply(lambda u: urlparse(u).netloc.lower())
print('Top source domains:')
print(df_domains['domain'].value_counts().head(20))

external = df_domains[~df_domains['domain'].str.contains('papalencyclicals.net|vatican.va', regex=True, na=False)]
print(f"\nExternal-domain records: {len(external)}")
if len(external):
    print(external[['pope', 'title', 'url']].head(25).to_string(index=False))

Top source domains:
domain
www.papalencyclicals.net      402
www.vatican.va                124
w2.vatican.va                  27
web.archive.org                 7
www.franciscan-archive.org      5
www.bluewaterarts.com           1
en.wikisource.org               1
www.ewtn.com                    1
www.nativeweb.org               1
www.adoremus.org                1
Name: count, dtype: int64

External-domain records: 17
                 pope                                                          title                                                                                                                                                                             url
    Pope Alexander IV                                         Clara claris praeclara                                                                                                                          http://www.franciscan-archive.org/bullarium/clara.html
      Pope Clement XI                             Latin 

In [9]:
# Review curated dead-link replacements and identify any remaining gaps
dead_domains = set(scraper.KNOWN_DEAD_DOMAINS)

dead_links = df_domains[df_domains['domain'].isin(dead_domains)].copy()
print(f"Dead-link records in index: {len(dead_links)}")

if DEAD_LINK_REPLACEMENTS_FILE.exists():
    df_replacements = pd.read_csv(DEAD_LINK_REPLACEMENTS_FILE).fillna('')
    usable = df_replacements[df_replacements['replacement_url'].str.strip().ne('')].copy()
    missing = df_replacements[df_replacements['replacement_url'].str.strip().eq('')].copy()

    print(f"Curated replacement rows: {len(df_replacements)}")
    print(f"Usable replacements: {len(usable)}")
    print(f"Still missing replacement URLs: {len(missing)}")

    preview_cols = ['doc_id', 'url', 'replacement_url']
    if len(usable):
        print("\nSample replacement mappings:")
        print(usable[preview_cols].head(15).to_string(index=False))

    if len(missing):
        print("\nRows still missing replacement URLs:")
        print(missing[preview_cols].head(15).to_string(index=False))
else:
    print(f"No curated replacement file found at {DEAD_LINK_REPLACEMENTS_FILE}")
    print("Run the domain audit above and create the CSV before scraping dead links.")

Dead-link records in index: 0
Curated replacement rows: 33
Usable replacements: 33
Still missing replacement URLs: 0

Sample replacement mappings:
                                   doc_id                                                url                                                                                                                  replacement_url
 pope_clement_xiii__accedamus_cum_fiducia    http://digilander.iol.it/magistero/c13acced.htm                         https://www.vatican.va/content/clemens-xiii/it/documents/breve-accedamus-cum-fiducia-25-giugno-1768.html
    pope_clement_xiii__pastoralis_officii    http://digilander.iol.it/magistero/c13pasto.htm                                                                                https://www.papalencyclicals.net/leo13/l13dul.htm
         pope_clement_xiii__quam_graviter    http://digilander.iol.it/magistero/c13quamg.htm                             https://www.vatican.va/content/clemens-xiii/it/documents/enciclica

In [10]:
# Apply curated replacements before scraping so the preview matches scraper behavior
replacement_map = load_dead_link_replacements()
documents, replaced_doc_ids = apply_dead_link_replacements(documents, replacement_map)

print(f"Documents using replacement URLs this run: {len(replaced_doc_ids)}")
if replaced_doc_ids:
    df_replaced = pd.DataFrame([
        {
            'doc_id': doc['doc_id'],
            'original_url': doc.get('original_url', ''),
            'url': doc.get('url', ''),
        }
        for doc in documents
        if doc['doc_id'] in replaced_doc_ids
    ])
    print(df_replaced.head(15).to_string(index=False))

2026-03-30 11:38:14,955 [INFO] Loaded 33 dead-link replacements from C:\Users\harrisrc\OneDrive - Chesterfield County VA\Documents\MSDS\encyclicals\data\processed\dead_link_replacements.csv


Documents using replacement URLs this run: 0


## Step 2: Download Document Texts

For each document in the index, we fetch the linked page and extract
the encyclical text. The scraper handles:
- HTML pages with text in the page body
- Pages that link to epub files
- Language detection (English, Latin, Italian, French)

In [11]:
# Identify papalencyclicals.net documents with suspiciously short text that
# may have been truncated by the old extraction logic (text in raw text nodes
# rather than <p> tags). Those raw files are deleted here so the scrape step
# below re-downloads them with the improved extractor.
#
# Set RESCRAPE_SHORT_DOCS = False to skip this step and keep cached files.
RESCRAPE_SHORT_DOCS = True
SHORT_TEXT_THRESHOLD = 3000  # chars; papalencyclicals.net docs below this may be truncated

df_scrape = pd.DataFrame(documents)
suspect_mask = (
    df_scrape['url'].fillna('').str.contains('papalencyclicals.net', regex=False)
    & df_scrape['text_length'].gt(0)
    & df_scrape['text_length'].lt(SHORT_TEXT_THRESHOLD)
    & df_scrape['language'].ne('unknown')
)
suspect_docs = df_scrape[suspect_mask].copy()
print(f"Potentially truncated papalencyclicals.net docs (text_length < {SHORT_TEXT_THRESHOLD}): {len(suspect_docs)}")
if len(suspect_docs):
    print(suspect_docs[['doc_id', 'text_length', 'url']].sort_values('text_length').to_string(index=False))

if RESCRAPE_SHORT_DOCS and len(suspect_docs):
    deleted = []
    for doc_id in suspect_docs['doc_id']:
        raw = RAW_DIR / f"{doc_id}.txt"
        if raw.exists():
            raw.unlink()
            deleted.append(doc_id)
    print(f"\nCleared {len(deleted)} cached files — they will be re-scraped below.")
elif not RESCRAPE_SHORT_DOCS:
    print("\nRESCRAPE_SHORT_DOCS=False — skipping cache clear.")

Potentially truncated papalencyclicals.net docs (text_length < 3000): 24
                                                       doc_id  text_length                                                                          url
                          pope_clement_xiii__in_dominico_agro          236                        https://www.papalencyclicals.net//clem13/c13indom.htm
                      church_councils__vatican_ii___1962_1965          246                      https://www.papalencyclicals.net//councils/vatican2.htm
                           pope_paul_vi__vatican_ii_documents          246                        https://www.papalencyclicals.net//paul06/vatican2.htm
                                 pope_pius_xi__rerum_condicio          400                  https://www.papalencyclicals.net//pius11/rerum-condicio.htm
                                 pope_st__pius_x__septimo_iam          400                     https://www.papalencyclicals.net//pius10/septimo-iam.htm
               

In [12]:
# Scrape all document texts (skips already-downloaded files)
# Use max_docs to limit for testing:
# documents = scrape_documents(documents, max_docs=100, show_progress=True, progress_label="Downloading")

documents = scrape_documents(documents, show_progress=True, progress_label="Downloading")
save_index(documents)
save_library_csv(documents)

2026-03-30 11:38:15,075 [INFO] Loaded 33 dead-link replacements from C:\Users\harrisrc\OneDrive - Chesterfield County VA\Documents\MSDS\encyclicals\data\processed\dead_link_replacements.csv


Downloading: 10/570 | saved=4 skipped=6 dead=0 errors=0

2026-03-30 11:38:41,919 [INFO] SSL verification failed in this environment; using insecure fallback (verify=False).
2026-03-30 11:39:29,397 [INFO] Failed to fetch https://www.papalencyclicals.net//ben14/annus-qui-hunc.htm after 3 attempts


Downloading: 132/570 | saved=6 skipped=125 dead=0 errors=1

2026-03-30 11:40:58,109 [INFO] Failed to fetch https://www.papalencyclicals.net//hon03/regula-e.htm after 2 attempts


...

2026-03-30 11:47:12,211 [INFO] Failed to fetch https://www.papalencyclicals.net//leo13/l13insig.htm after 2 attempts


Downloading: 287/570 | saved=16 skipped=268 dead=0 errors=3

2026-03-30 11:48:09,443 [INFO] Failed to fetch https://www.papalencyclicals.net//leo13/l13qunos.htm after 3 attempts


Downloading: 570/570 | saved=29 skipped=537 dead=0 errors=4

2026-03-30 11:52:55,564 [INFO] Saved index with 570 documents to C:\Users\harrisrc\OneDrive - Chesterfield County VA\Documents\MSDS\encyclicals\data\encyclicals_index.json
2026-03-30 11:52:55,576 [INFO] Saved LIBRARY.csv with 570 rows


In [13]:
# Summary
df = pd.DataFrame(documents)
print(f"Total documents: {len(df)}")
print(f"\nBy language:")
print(df['language'].value_counts())
print(f"\nBy format:")
print(df['format'].value_counts())
print(f"\nDocuments with text: {(df['text_length'] > 100).sum()}")

Total documents: 570

By language:
language
en         559
unknown      5
la           4
             2
Name: count, dtype: int64

By format:
format
html     566
error      4
Name: count, dtype: int64

Documents with text: 566


In [14]:
# List raw text files
raw_files = list(RAW_DIR.glob('*.txt'))
print(f"Raw text files on disk: {len(raw_files)}")
sizes = [f.stat().st_size for f in raw_files]
print(f"Total size: {sum(sizes)/1024/1024:.1f} MB")
print(f"Average size: {sum(sizes)/len(sizes)/1024:.1f} KB")

Raw text files on disk: 598
Total size: 21.2 MB
Average size: 36.3 KB
